# CineCue: 3. Features and chronological splits

Run notebooks 1 and 2 first. We create a small feature set and split the interactions
in time order. Recommender training, evaluation, re-ranking, and explanation generation
belong to Phase 2.

| Field | Meaning and missing-value handling |
| --- | --- |
| `interaction_id`, `source_row` | Traceability only; never model inputs. |
| `user_id`, `movie_id` | Positive integer keys, used as categorical identities. |
| `rating` | Target: original whole-star integer from 1 to 5. |
| `timestamp`, `rated_at` | Unix seconds and UTC submission time; retained for ordering and audit. |
| `release_year` | Final four-digit year in parentheses in a movie title, within 1800–2003; nullable if unsupported. |
| `release_year_missing` | 1 when no supported year is available, otherwise 0. |
| `genre_*` | One 0/1 column for each official genre; a movie can have several. |
| `hour_utc` | Submission hour, 0–23; a categorical context feature, not local watch time. |
| `weekday_utc` | Submission weekday, Monday=0 through Sunday=6. |
| `is_weekend_utc` | 1 for Saturday or Sunday in UTC. |

Missing required IDs, ratings, metadata, and times were handled explicitly in cleaning.
We do not invent session boundaries. Any future session feature must be optional and
described as a *rating-session proxy*, not a viewing session.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Work from either the repository root or its notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "requirements.txt").is_file(), "Open this notebook from the cloned repository."
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 20)

ratings = pd.read_csv(PROCESSED / "ratings_clean.csv")
ratings["rated_at"] = pd.to_datetime(ratings["rated_at"], utc=True)
movies = pd.read_csv(PROCESSED / "movies_clean.csv")
assert ratings.rated_at.notna().all() and movies.movie_id.is_unique

In [2]:
# Match a year only at the end of the title; other numbers are not release years.
years = pd.to_numeric(movies.title.str.extract(r"\((\d{4})\)\s*$", expand=False), errors="coerce")
release_year = years.where(years.between(1800, 2003)).astype("Int64")
genre_flags = movies.genres.str.get_dummies(sep="|").astype("int8")
genre_flags.columns = ["genre_" + col.lower().replace("'", "").replace("-", "_").replace(" ", "_")
                       for col in genre_flags.columns]
assert genre_flags.columns.is_unique
movie_features = pd.concat([movies[["movie_id"]], release_year.rename("release_year"),
                           release_year.isna().astype("int8").rename("release_year_missing"), genre_flags], axis=1)
features = ratings.merge(movie_features, on="movie_id", how="left", validate="many_to_one", indicator=True)
assert features["_merge"].eq("both").all() and len(features) == len(ratings)
features = features.drop(columns="_merge")
features["hour_utc"] = features.rated_at.dt.hour.astype("int8")
features["weekday_utc"] = features.rated_at.dt.dayofweek.astype("int8")
features["is_weekend_utc"] = features.weekday_utc.ge(5).astype("int8")
features = features.sort_values(["timestamp", "source_row"], kind="stable").reset_index(drop=True)
print("Movie features:", movie_features.shape)
print("Interaction features:", features.shape)
print("Movies missing a supported release year:", int(release_year.isna().sum()))
print("Rating rows missing a supported release year:", int(features.release_year.isna().sum()))
display(features.dtypes.astype(str).rename("dtype").to_frame())

Movie features: (3883, 21)
Interaction features: (1000209, 30)
Movies missing a supported release year: 0
Rating rows missing a supported release year: 0


,dtype
interaction_id,int64
user_id,int64
movie_id,int64
rating,int64
timestamp,int64
source_row,int64
rated_at,"datetime64[ns, UTC]"
release_year,Int64
release_year_missing,int8
genre_action,int8


## Scaling and leakage

IDs identify categories and must not be scaled. Multihot genre columns and binary flags
are already 0/1. Hour and weekday are categorical context codes, so we do not normalize
them as continuous measurements. The rating target stays on its original scale.
Release year stays as a nullable integer metadata field; no continuous model or transformer
is needed in Phase 1. A future model that needs a scaled or imputed year must fit those
transformers on training rows only, then apply them unchanged to later splits.

We use the known catalog to make fixed genre flags. Movie availability dates are not
provided, so historical catalog availability cannot be reconstructed from these files.
Release year does not prove when a title entered MovieLens. Catalog metadata may need
additional availability checks in a real deployment. No rating-derived popularity,
user averages, or full-data EDA aggregates enter these features. Future history features
must use events strictly before the event being predicted, including within training.

## A global time split

We aim for cumulative 80% and 90% boundaries. For each boundary, select the timestamp
group whose cumulative size is closest to the target; an equal-distance tie chooses
the earlier group. All events at that timestamp stay together. This is deterministic
and reproducible without randomness. Validation and test keep unseen users and movies
so Phase 2 can handle and report cold starts instead of silently discarding them.

In [3]:
timestamp_counts = features.groupby("timestamp", sort=True).size()
cumulative = timestamp_counts.cumsum()
assert len(cumulative) >= 3, "At least three distinct timestamps are required."
# Leave at least two groups after training, and at least one after validation.
train_candidates = cumulative.iloc[:-2]
train_cutoff = int((train_candidates - 0.80 * len(features)).abs().idxmin())
validation_candidates = cumulative.loc[cumulative.index > train_cutoff].iloc[:-1]
validation_cutoff = int((validation_candidates - 0.90 * len(features)).abs().idxmin())
splits = {
    "train": features.loc[features.timestamp.le(train_cutoff)].copy(),
    "validation": features.loc[features.timestamp.gt(train_cutoff) & features.timestamp.le(validation_cutoff)].copy(),
    "test": features.loc[features.timestamp.gt(validation_cutoff)].copy(),
}
split_rows = []
for name, frame in splits.items():
    assert len(frame) > 0
    split_rows.append({"split": name, "rows": len(frame), "percentage": 100 * len(frame) / len(features),
                       "start_utc": str(frame.rated_at.min()), "end_utc": str(frame.rated_at.max()),
                       "users": int(frame.user_id.nunique()), "movies": int(frame.movie_id.nunique())})
split_table = pd.DataFrame(split_rows).set_index("split")
display(split_table)
print("Training cutoff (inclusive):", train_cutoff, pd.to_datetime(train_cutoff, unit="s", utc=True))
print("Validation cutoff (inclusive):", validation_cutoff, pd.to_datetime(validation_cutoff, unit="s", utc=True))

train, validation, test = (splits[name] for name in ["train", "validation", "test"])
assert train.timestamp.max() < validation.timestamp.min()
assert validation.timestamp.max() < test.timestamp.min()
combined = pd.concat(splits.values(), ignore_index=True)
assert len(combined) == len(features)
assert combined.interaction_id.is_unique
assert set(combined.interaction_id) == set(features.interaction_id)
pd.testing.assert_frame_equal(combined.sort_values("interaction_id").reset_index(drop=True),
                              features.sort_values("interaction_id").reset_index(drop=True))
assert combined.rating.between(1, 5).all() and combined.rating.mod(1).eq(0).all()
assert movie_features.movie_id.is_unique
assert movie_features[genre_flags.columns].isin([0, 1]).all().all()
assert features.hour_utc.between(0, 23).all() and features.weekday_utc.between(0, 6).all()
print("Passed: no row loss, duplicated interactions, temporal overlap, or timestamp groups crossing splits.")

,rows,percentage,start_utc,end_utc,users,movies
split,,,,,,
train,800168,80.00008,2000-04-25 23:05:32+00:00,2000-12-02 14:52:18+00:00,5400,3662
validation,100020,9.99991,2000-12-02 14:52:28+00:00,2000-12-29 23:42:47+00:00,1134,3305
test,100021,10.00001,2000-12-29 23:43:34+00:00,2003-02-28 17:49:50+00:00,1209,3407


Training cutoff (inclusive): 975768738 2000-12-02 14:52:18+00:00
Validation cutoff (inclusive): 978133367 2000-12-29 23:42:47+00:00


Passed: no row loss, duplicated interactions, temporal overlap, or timestamp groups crossing splits.


In [4]:
def unseen_counts(frame, reference, split_name, reference_name):
    unknown_users = ~frame.user_id.isin(reference.user_id)
    unknown_movies = ~frame.movie_id.isin(reference.movie_id)
    return {"split": split_name, "reference": reference_name,
            "unseen_users": int(frame.loc[unknown_users, "user_id"].nunique()),
            "unseen_movies": int(frame.loc[unknown_movies, "movie_id"].nunique()),
            "rows_unseen_user": int(unknown_users.sum()), "rows_unseen_movie": int(unknown_movies.sum()),
            "rows_either_unseen": int((unknown_users | unknown_movies).sum())}

cold_start = [unseen_counts(validation, train, "validation", "train"),
              unseen_counts(test, train, "test", "train"),
              unseen_counts(test, pd.concat([train, validation]), "test", "train + validation")]
display(pd.DataFrame(cold_start))
display(Markdown(f"**Interpretation:** Relative to training, validation has "
                 f"{cold_start[0]['unseen_users']:,} unseen users and {cold_start[0]['unseen_movies']:,} unseen movies; "
                 f"test has {cold_start[1]['unseen_users']:,} unseen users and "
                 f"{cold_start[1]['unseen_movies']:,} unseen movies. "
                 "Phase 2 needs fallback predictions for these cases. The extra train + validation "
                 "comparison describes a possible final refit, not permission to train on validation during tuning."))

,split,reference,unseen_users,unseen_movies,rows_unseen_user,rows_unseen_movie,rows_either_unseen
0,validation,train,611,16,76176,17,76180
1,test,train,200,33,19326,98,19413
2,test,train + validation,29,28,4209,90,4298


**Interpretation:** Relative to training, validation has 611 unseen users and 16 unseen movies; test has 200 unseen users and 33 unseen movies. Phase 2 needs fallback predictions for these cases. The extra train + validation comparison describes a possible final refit, not permission to train on validation during tuning.

In [5]:
movie_features.to_csv(PROCESSED / "movie_features.csv", index=False)
features.to_csv(PROCESSED / "interaction_features.csv", index=False)
for name, frame in splits.items():
    frame.to_csv(PROCESSED / f"{name}.csv", index=False)
    # Read each artifact back to catch write/schema/count problems.
    saved = pd.read_csv(PROCESSED / f"{name}.csv", dtype={"release_year": "Int64"})
    assert len(saved) == len(frame) and list(saved.columns) == list(frame.columns)
    assert saved.interaction_id.tolist() == frame.interaction_id.tolist()
    assert saved.rating.tolist() == frame.rating.tolist()
    assert pd.to_datetime(saved.rated_at, utc=True).equals(frame.rated_at.reset_index(drop=True))
summary = {"total_rows": len(features), "train_cutoff_unix": train_cutoff,
           "validation_cutoff_unix": validation_cutoff, "splits": split_rows, "cold_start": cold_start,
           "genre_columns": list(genre_flags.columns),
           "missing_release_year_movies": int(release_year.isna().sum()),
           "missing_release_year_interactions": int(features.release_year.isna().sum())}
(PROCESSED / "split_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
display(Markdown(f"**Saved:** movie_features.csv, interaction_features.csv, train.csv, validation.csv, "
                 f"test.csv, and split_summary.json in data/processed/. "
                 f"The splits contain {len(train):,}, {len(validation):,}, and {len(test):,} rows. "
                 "All generated data stays local and is excluded from Git."))

**Saved:** movie_features.csv, interaction_features.csv, train.csv, validation.csv, test.csv, and split_summary.json in data/processed/. The splits contain 800,168, 100,020, and 100,021 rows. All generated data stays local and is excluded from Git.

CSV reload reminder: read `rated_at` with `pd.to_datetime(..., utc=True)`, read
`release_year` as nullable `Int64`, and treat IDs and hour/weekday codes as categories
when a future model requires that representation. Fit encoders and any learned
transformations on training data only, with explicit handling for unseen values.

Phase 1 ends here. We have not trained or evaluated a recommender or generated explanations.